In [ ]:
!pip install sentence-transformers faiss-cpu transformers

In [ ]:
import torch

# ⚡ AUTO-DETECT GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Using device: {device}")

if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("   ⚠️ No GPU detected — running on CPU (will be slower)")

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    
    return chunks

print("✅ Imports and Tools loaded!")

In [ ]:
import sqlite3
import re

conn = sqlite3.connect('rag_database.db')
cursor = conn.cursor()

cursor.execute("SELECT id, subject, body FROM emails")
saved_emails = cursor.fetchall()
conn.close()

def clean_email(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'---------- Forwarded message ---------.*', '', text, flags=re.DOTALL)
    text = re.sub(r'On .* wrote:', '', text)
    text = re.sub(r'\n+', '\n', text)
    return text.strip()

master_chunk_list = []
master_metadata_list = []

for email in saved_emails:
    email_id, subject, raw_body = email
    
    clean_body = clean_email(raw_body)
    
    full_text = f"Subject: {subject}\nBody: {clean_body}"
    
    email_chunks = chunk_text(full_text, chunk_size=300, overlap=50)
    
    for chunk in email_chunks:
        master_chunk_list.append(chunk)
        master_metadata_list.append({"id": email_id, "subject": subject})

print(f"✅ Extracted {len(master_chunk_list)} chunks!")

In [ ]:
from sentence_transformers import SentenceTransformer

# ⚡ Load embedding model on GPU
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

doc_embeddings = model.encode(
    master_chunk_list,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(doc_embeddings.shape)

In [ ]:
import faiss
import numpy as np 

doc_embeddings = np.array(doc_embeddings).astype('float32')

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(doc_embeddings)

print(f"Number of documents {index.ntotal}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import faiss
import numpy as np

print("🧠 Loading FLAN-T5 Generator...")

model_name = "google/flan-t5-base"

# ⚡ FIX 1: Removed force_download=True (was re-downloading ~1GB every run!)
tokenizer = AutoTokenizer.from_pretrained(model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ⚡ FIX 2: Move model to GPU for 10-20x faster generation
gen_model = gen_model.to(device)
gen_model.eval()


def ask_my_emails(question, top_k=3):
    print(f"\n🤔 Question: {question}")

    # =========================
    # 1. RETRIEVAL (COSINE FAISS)
    # =========================
    question_embedding = model.encode([question]).astype('float32')
    faiss.normalize_L2(question_embedding)

    distances, indices = index.search(question_embedding, top_k)

    retrieved_chunks = []
    print("\n🔍 Retrieved Context:")

    for i in indices[0]:
        chunk = master_chunk_list[i]
        retrieved_chunks.append(chunk)
        print(f"  - {chunk[:80]}...")

    context = "\n".join(retrieved_chunks)[:1800]

    # =========================
    # 2. BETTER RAG PROMPT (IMPORTANT)
    # =========================

    prompt = f"""
You are a helpful AI assistant that answers questions based ONLY on the provided context.

If the answer is not in the context, say "I don't have enough information in the emails."

Context:
{context}

Question:
{question}

Answer clearly and concisely:
"""

    # =========================
    # 3. TOKENIZATION
    # =========================
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)  # ⚡ FIX 3: Move inputs to GPU too

    # =========================
    # 4. GENERATION (OPTIMIZED)
    # =========================
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_length=200,
            num_beams=5,
            temperature=0.7,
            do_sample=False,
            repetition_penalty=1.2
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # =========================
    # 5. OUTPUT
    # =========================
    print("\n" + "=" * 50)
    print(f"🤖 AI Answer: {answer}")


    
    print("=" * 50)

    return answer


print(f"✅ Upgraded FLAN-T5 RAG Ready! (on {device})")

In [ ]:
ask_my_emails("when is vtop downtime", 3)